# PatchTST CI vs CD -- ECL

Trains channel-independent (CI) and channel-dependent (CD) variants of PatchTST at
pred_len in [96, 336] on ECL (321 variates, hourly) with identical hyperparameters and seed.
Results are saved to `results/ci_cd_ecl.csv`.

Split: 15840 train / 5256 val / 5256 test rows (Liu et al., iTransformer, ICLR 2024).

Memory note: CD encoder receives C*N = 321*11 = 3531 tokens at seq_len=96. The attention
matrix at B=4, H=16 occupies ~3.2 GB; batch_size_cd=4 is required to stay within T4 budget.
CI uses batch_size_ci=128 since it processes variates independently.

In [ ]:
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## Dataset

In [ ]:
class ECLDataset(Dataset):
    """ECL (Electricity Consuming Load) multivariate dataset.

    321 client-level hourly electricity consumption variates.
    Split: 15840 train / 5256 val / 5256 test rows.
    Normalization: z-score per channel, scaler fit on train split only.

    Source: thuml/iTransformer repo, datasets/electricity.csv.
    Split matches Liu et al., iTransformer, ICLR 2024.

    Args:
        csv_path: Path to electricity.csv.
        split: One of 'train', 'val', 'test'.
        seq_len: Number of input timesteps per sample.
        pred_len: Number of target timesteps immediately following the input.
    """

    _TRAIN_END = 15840
    _VAL_END = 15840 + 5256    # 21096
    _TEST_END = 15840 + 5256 + 5256  # 26352

    def __init__(self, csv_path: str, split: str, seq_len: int, pred_len: int) -> None:
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be one of 'train', 'val', 'test', got '{split}'.")

        df = pd.read_csv(csv_path, usecols=lambda c: c != "date")

        if len(df) < self._TEST_END:
            raise ValueError(f"ECL CSV has {len(df)} rows; expected at least {self._TEST_END}.")

        train_df = df.iloc[: self._TRAIN_END]
        self._mean: np.ndarray = train_df.mean(axis=0).to_numpy(dtype=np.float32)
        self._std: np.ndarray = train_df.std(axis=0, ddof=0).clip(lower=1e-8).to_numpy(dtype=np.float32)

        normalized = (df.values.astype(np.float32) - self._mean) / self._std

        if split == "train":
            self._data = normalized[: self._TRAIN_END]
        elif split == "val":
            self._data = normalized[self._TRAIN_END : self._VAL_END]
        else:
            self._data = normalized[self._VAL_END : self._TEST_END]

        self.seq_len = seq_len
        self.pred_len = pred_len

    def __len__(self) -> int:
        return len(self._data) - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)


## Model

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_size: int, stride: int, d_model: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.projection = nn.Linear(patch_size, d_model)
        self.dropout = nn.Dropout(dropout)
        self._d_model = d_model

    def _sinusoidal_pe(self, num_patches: int, device: torch.device) -> torch.Tensor:
        position = torch.arange(num_patches, device=device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, self._d_model, 2, device=device) * (-math.log(10000.0) / self._d_model)
        )
        pe = torch.zeros(num_patches, self._d_model, device=device)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.squeeze(-1).unfold(dimension=-1, size=self.patch_size, step=self.stride)
        x = self.projection(x)
        pe = self._sinusoidal_pe(x.shape[1], x.device)
        return self.dropout(x + pe)


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x = x + attn_out
        return x + self.ff(self.norm2(x))


class TransformerEncoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class ForecastHead(nn.Module):
    def __init__(self, num_patches: int, d_model: int, pred_len: int, dropout: float) -> None:
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self.dropout(x.flatten(1)))


class PatchTST(nn.Module):
    """PatchTST with CI and CD mode support.

    channel_mixing=False (CI): each variate processed independently; encoder sees N
    patch tokens per channel with weights shared across channels by construction.
    channel_mixing=True  (CD): patches from all variates concatenated along the sequence
    dimension before the encoder; attention runs over C*N tokens simultaneously.

    Reference: Nie et al., "A Time Series Is Worth 64 Words", ICLR 2023.
               https://arxiv.org/abs/2211.14730
    """

    def __init__(
        self,
        seq_len: int,
        pred_len: int,
        num_variates: int,
        patch_size: int = 16,
        stride: int = 8,
        d_model: int = 128,
        num_heads: int = 16,
        num_layers: int = 3,
        dropout: float = 0.2,
        channel_mixing: bool = False,
    ) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
        self.num_variates = num_variates
        self.channel_mixing = channel_mixing
        self.num_patches = (seq_len - patch_size) // stride + 1
        self.embedding = PatchEmbedding(patch_size=patch_size, stride=stride, d_model=d_model, dropout=dropout)
        self.encoder = TransformerEncoder(d_model=d_model, num_heads=num_heads, num_layers=num_layers, dropout=dropout)
        self.head = ForecastHead(num_patches=self.num_patches, d_model=d_model, pred_len=pred_len, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, L, 1)
        x = self.embedding(x)  # (B*C, N, D)
        if self.channel_mixing:
            x = x.reshape(B, C * self.num_patches, -1)  # (B, C*N, D)
            x = self.encoder(x)                          # (B, C*N, D)
            x = x.reshape(B * C, self.num_patches, -1)  # (B*C, N, D)
        else:
            x = self.encoder(x)  # (B*C, N, D)
        x = self.head(x)  # (B*C, pred_len)
        return x.reshape(B, C, -1).permute(0, 2, 1)  # (B, pred_len, C)


## Training Infrastructure

In [ ]:
class EarlyStopping:
    """Stop training when validation MSE stops improving.

    Args:
        patience: Number of epochs to wait after last improvement.
        checkpoint_path: Path to save the best model state dict.
    """

    def __init__(self, patience: int = 10, checkpoint_path: str = "best_model.pt") -> None:
        self.patience = patience
        self.checkpoint_path = checkpoint_path
        self.best_val_mse = float("inf")
        self.counter = 0
        self.best_epoch = 0

    def step(self, val_mse: float, model: nn.Module, epoch: int) -> bool:
        """Save checkpoint if improved; increment counter otherwise.

        Args:
            val_mse: Validation MSE for the current epoch.
            model: Model whose state dict to checkpoint on improvement.
            epoch: Current epoch number (1-indexed).

        Returns:
            True if patience is exhausted and training should stop.
        """
        if val_mse < self.best_val_mse:
            self.best_val_mse = val_mse
            self.counter = 0
            self.best_epoch = epoch
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
        return self.counter >= self.patience


def compute_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple[float, float]:
    """Compute MSE and MAE between predictions and targets.

    Args:
        pred: Predicted tensor, any shape.
        target: Ground truth tensor, same shape as pred.

    Returns:
        Tuple of (mse, mae) as Python floats.
    """
    mse = torch.mean((pred - target) ** 2).item()
    mae = torch.mean(torch.abs(pred - target)).item()
    return mse, mae


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
) -> tuple[float, float]:
    """Run one training epoch and return mean MSE and MAE over all batches.

    Args:
        model: The model to train.
        loader: Training DataLoader.
        optimizer: Optimizer instance.
        criterion: Loss function (MSELoss).

    Returns:
        Tuple of (mean_mse, mean_mae) weighted by batch size.
    """
    model.train()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(x)
        loss = criterion(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        _, mae = compute_metrics(pred.detach(), y)
        batch = x.size(0)
        total_mse += loss.item() * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    """Evaluate model on a DataLoader and return mean MSE and MAE.

    Args:
        model: The model to evaluate.
        loader: Validation or test DataLoader.

    Returns:
        Tuple of (mean_mse, mean_mae) weighted by batch size.
    """
    model.eval()
    total_mse, total_mae, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x)
        mse, mae = compute_metrics(pred, y)
        batch = x.size(0)
        total_mse += mse * batch
        total_mae += mae * batch
        n += batch
    return total_mse / n, total_mae / n


## run_mode

In [ ]:
def run_mode(mode: str, csv_path: str, config: dict, results_dir: Path, ckpt_dir: Path) -> dict:
    """Train one CI/CD run for ECL and return a result dict.

    CD mode uses a reduced batch size due to memory constraints on the T4.
    With seq_len=96, patch_size=16, stride=8: N=11 patches per variate.
    CD encoder receives C*N = 321*11 = 3531 tokens. At H=16 heads, the attention
    matrix occupies B * 16 * 3531^2 * 4 bytes; batch_size=4 uses ~3.2 GB.
    CI processes variates independently so memory is linear in C, not quadratic.

    Args:
        mode: 'CI' or 'CD'.
        csv_path: Path to electricity.csv.
        config: Dict containing all hyperparameters plus 'pred_len'.
        results_dir: Directory for output CSVs.
        ckpt_dir: Directory for model checkpoints.

    Returns:
        Dict with keys: mode, pred_len, test_mse, test_mae, best_val_mse,
        best_epoch, num_params, batch_size, seed.
    """
    channel_mixing = mode == "CD"
    pred_len = config["pred_len"]
    batch_size = config["batch_size_cd"] if channel_mixing else config["batch_size_ci"]
    ckpt_path = str(ckpt_dir / f"patchtst_ecl_{mode.lower()}_pred{pred_len}.pt")

    torch.manual_seed(config["seed"])
    random.seed(config["seed"])
    np.random.seed(config["seed"])

    train_ds = ECLDataset(csv_path, "train", config["seq_len"], pred_len)
    val_ds = ECLDataset(csv_path, "val", config["seq_len"], pred_len)
    test_ds = ECLDataset(csv_path, "test", config["seq_len"], pred_len)

    loader_kwargs: dict = {"batch_size": batch_size, "num_workers": 2, "pin_memory": True}
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    model = PatchTST(
        seq_len=config["seq_len"],
        pred_len=pred_len,
        num_variates=config["num_variates"],
        patch_size=config["patch_size"],
        stride=config["stride"],
        d_model=config["d_model"],
        num_heads=config["num_heads"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        channel_mixing=channel_mixing,
    ).to(DEVICE)

    total_params = sum(p.numel() for p in model.parameters())
    print(
        f"[{mode} pred={pred_len}] Training | params: {total_params:,} | "
        f"batch_size: {batch_size} | CD tokens per sample: {config['num_variates'] * model.num_patches}"
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-4)
    warmup_epochs = config["warmup_epochs"]

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, config["epochs"] - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.MSELoss()
    early_stopping = EarlyStopping(patience=config["patience"], checkpoint_path=ckpt_path)
    t0 = time.time()

    for epoch in range(1, config["epochs"] + 1):
        train_mse, _ = train_one_epoch(model, train_loader, optimizer, criterion)
        val_mse, _ = evaluate(model, val_loader)
        scheduler.step()
        if epoch % 5 == 0 or epoch == 1:
            lr_now = optimizer.param_groups[0]["lr"]
            print(
                f"  [{mode} pred={pred_len}] Epoch {epoch:3d}/{config['epochs']} | "
                f"train MSE {train_mse:.4f} | val MSE {val_mse:.4f} | "
                f"lr {lr_now:.2e} | {time.time() - t0:.0f}s"
            )
        if early_stopping.step(val_mse, model, epoch):
            print(f"  [{mode} pred={pred_len}] Early stop at epoch {epoch}. Best: epoch {early_stopping.best_epoch}.")
            break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    test_mse, test_mae = evaluate(model, test_loader)
    print(
        f"  [{mode} pred={pred_len}] Test MSE: {test_mse:.4f} | "
        f"Test MAE: {test_mae:.4f} | "
        f"Best val MSE: {early_stopping.best_val_mse:.4f} @ epoch {early_stopping.best_epoch}"
    )
    return {
        "mode": mode, "pred_len": pred_len,
        "test_mse": round(test_mse, 6), "test_mae": round(test_mae, 6),
        "best_val_mse": round(early_stopping.best_val_mse, 6),
        "best_epoch": early_stopping.best_epoch,
        "num_params": total_params, "batch_size": batch_size, "seed": config["seed"],
    }


## Config

In [ ]:
CSV_PATH = "/kaggle/input/long-horizon-datasets/ECL/electricity.csv"
RESULTS_DIR = Path("results")
CKPT_DIR = Path("results/checkpoints")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# seq_len=96: 321 variates at seq_len=512 exceeds T4 memory in CD mode (C*N=321*63=20223 tokens).
# batch_size_cd=4: attention matrix at C*N=3531 tokens uses ~3.2 GB at B=4, H=16.
# batch_size_ci=128: CI processes variates independently; memory is linear in C.
# All other hyperparameters are identical across CI and CD.
BASE_CONFIG = {
    "seq_len": 96,
    "num_variates": 321,
    "patch_size": 16,
    "stride": 8,
    "d_model": 128,
    "num_heads": 16,
    "num_layers": 3,
    "dropout": 0.2,
    "lr": 1e-4,
    "warmup_epochs": 10,
    "batch_size_ci": 128,
    "batch_size_cd": 4,
    "epochs": 100,
    "patience": 10,
    "seed": 42,
}

PRED_LENS = [96, 336]


## Run CI and CD

In [ ]:
all_results = []

for pred_len in PRED_LENS:
    config = {**BASE_CONFIG, "pred_len": pred_len}
    for mode in ["CI", "CD"]:
        print(f'\n{"=" * 60}')
        print(f"Mode: {mode} | pred_len: {pred_len}")
        print(f'{"=" * 60}')
        all_results.append(run_mode(mode, CSV_PATH, config, RESULTS_DIR, CKPT_DIR))

results_df = pd.DataFrame(all_results)
results_path = RESULTS_DIR / "ci_cd_ecl.csv"
results_df.to_csv(results_path, index=False)

print("\n=== CI vs CD Results -- ECL ===")
print(results_df[["mode", "pred_len", "test_mse", "test_mae", "best_epoch"]].to_string(index=False))

print("\n--- MSE ratio (CD/CI) per horizon ---")
pivot = results_df.pivot(index="pred_len", columns="mode", values="test_mse")
pivot["ratio_cd_ci"] = pivot["CD"] / pivot["CI"]
pivot["winner"] = np.where(pivot["CI"] < pivot["CD"], "CI", "CD")
print(pivot[["CI", "CD", "ratio_cd_ci", "winner"]].to_string())

print(f"\nResults saved to {results_path}")


## Verify Output Files

In [ ]:
required = [RESULTS_DIR / "ci_cd_ecl.csv"]

if not all(p.exists() for p in required):
    missing = [p for p in required if not p.exists()]
    raise RuntimeError(f"Output files missing: {missing}. Do not close the session.")
print("All output files verified.")
